# Track B — Task 5: LoRA & Last-layer Fine-tuning (TRACK_B_ARAHAN_V3.md B8)

**BDC Satria Data 2026** | Paradigma BEDA dari notebook 07 (head di atas embedding beku):
di sini sebagian vision tower SigLIP ikut dilatih.

> ⚠️ **Prioritas paling rendah (C: Task 7).** Kerjakan HANYA kalau Task 1-6 sudah aman.
> Kalau jadwal geser, INI yang dikorbankan — bukan head grid.

| Varian | Parameter awal B8 | Alasan |
|---|---|---|
| `lora` | rank 8, alpha 16, LR **1e-4**, epoch 3-5 | adapter kecil, backbone asli 100% beku |
| `last_layer` | buka 1 blok terakhir, head LR 3e-4, backbone LR /50 | LLRD — dulu satu-satunya yang membantu (+0.0158) |

**Hanya vision tower.** `SiglipModel` punya `text_model` + `vision_model`; kompetisi melarang
informasi di luar konten gambar. LoRA discoped lewat regex `VISION_ATTN_PATTERN` —
kalau `target_modules` dikirim sebagai list biasa, peft juga menempel ke text tower
(terbukti empiris: 48 vision + 48 text). Sudah dikunci test.

---
## 🔧 SETUP

In [ ]:
# Cell 1 -- Repo + dependensi (peft WAJIB) + Drive
import os
if not os.path.exists('/content/satria-data-bdcugm02'):
    !git clone https://github.com/agaggigit/satria-data-bdcugm02.git
else:
    !git -C /content/satria-data-bdcugm02 pull

!pip install -q peft transformers scikit-learn

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import torch
assert torch.cuda.is_available(), 'GPU tidak aktif -- Task 5 butuh GPU (Runtime > Change runtime type > T4)'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Cell 2 -- sys.path ke src/ dan experiments/
import sys
sys.path.insert(0, '/content/satria-data-bdcugm02/track_b/src')
sys.path.insert(0, '/content/satria-data-bdcugm02/track_b/experiments')

from config import CFG, make_cfg
import lora_ft
print('LORA: rank', lora_ft.LORA_RANK, 'alpha', lora_ft.LORA_ALPHA, 'lr', lora_ft.LORA_LR,
      'epochs', lora_ft.LORA_EPOCHS)
print('LAST_LAYER: n_blocks', lora_ft.LAST_LAYER_N_BLOCKS, 'head_lr', lora_ft.LAST_LAYER_HEAD_LR,
      'backbone_ratio', lora_ft.LAST_LAYER_BACKBONE_LR_RATIO)

---
## 🧪 SMOKE TEST MEKANIKA (CPU, detik) — jalankan SEBELUM bakar GPU

Membuktikan loop-nya benar: gradient sampai ke parameter yang benar, optimizer meng-update,
tidak ada NaN. Pakai `SMOKE_LR` (sengaja > LR produksi B8) — LR produksi kecil ON PURPOSE
untuk melindungi representasi pretrained, jadi butuh ribuan step untuk hafal 1 batch.
Memisahkan dua hal ini persis seperti `sanity_overfit.py` di Fase 0.

In [ ]:
# Cell 3 -- smoke test mekanika kedua varian (encoder KECIL palsu, CPU, tanpa unduh bobot)
import torch.nn as nn
from lora_ft import SMOKE_LR, smoke_test_loop

D_FAKE, N_LAYERS_FAKE = 16, 4

class _Out:
    def __init__(self, p): self.pooler_output = p

class _L(nn.Module):
    def __init__(s, d):
        super().__init__()
        s.self_attn = nn.Module()
        for nm in ('q_proj', 'k_proj', 'v_proj', 'out_proj'):
            setattr(s.self_attn, nm, nn.Linear(d, d))
        s.mlp = nn.Linear(d, d)
    def forward(s, x):
        a = s.self_attn.q_proj(x) + s.self_attn.v_proj(x) + s.self_attn.k_proj(x)
        return s.mlp(s.self_attn.out_proj(a))

class _Tower(nn.Module):
    def __init__(s, d, n):
        super().__init__()
        s.encoder = nn.Module()
        s.encoder.layers = nn.ModuleList([_L(d) for _ in range(n)])
    def forward(s, pixel_values):
        x = pixel_values
        for l in s.encoder.layers: x = l(x)
        return _Out(x)

class _FakeSiglip(nn.Module):
    def __init__(s, d, n):
        super().__init__()
        s.vision_model, s.text_model = _Tower(d, n), _Tower(d, n)

factory = lambda: _FakeSiglip(D_FAKE, N_LAYERS_FAKE)
for variant in ('lora', 'last_layer'):
    fl = smoke_test_loop(variant, factory, hidden_size=D_FAKE, n_steps=200,
                         device='cpu', target_loss=0.2)
    print(f'{variant:12s} smoke_lr={SMOKE_LR[variant]:<8.0e} final_loss={fl:.6f}  OK')
print('\nMEKANIKA LOOP TERBUKTI BENAR -- boleh lanjut ke GPU')

---
## 🔥 SMOKE TEST FOLD 0 (GPU, 2 epoch) — B8: catat waktu/epoch DULU

Jangan langsung 5-fold. Ukur `minutes_per_epoch` di fold 0, hitung estimasi biaya penuh,
baru putuskan lanjut atau tidak.

In [ ]:
# Cell 4 -- fold 0, gambar asli, 2 epoch. Ganti VARIANT untuk mencoba yang lain.
from lora_ft import run_smoke_test_fold0

VARIANT = 'lora'          # 'lora' atau 'last_layer'
CHECKPOINT = 'google/siglip2-base-patch16-256'

cfg = make_cfg(run_name=f'{VARIANT}_ft_v3', batch=32)
result = run_smoke_test_fold0(VARIANT, cfg, CHECKPOINT, max_epochs=2)
result

In [ ]:
# Cell 5 -- keputusan lanjut/stop berdasarkan biaya GPU nyata (bukan tebakan)
mins = result['minutes_per_epoch']
epochs_full = lora_ft.LORA_EPOCHS if VARIANT == 'lora' else 5
est_hours = mins * epochs_full * 5 / 60          # 5 fold

print(f"variant           : {VARIANT}")
print(f"val_f1 (2 epoch)  : {result['val_f1']:.4f}")
print(f"minutes_per_epoch : {mins:.2f}")
print(f"estimasi 5-fold x {epochs_full} epoch = {est_hours:.1f} jam GPU")
print()
print('Baseline yang harus dikalahkan: CV mean 0.9901 (siglip2so400m + kNN).')
print('Kalau val_f1 2-epoch masih jauh di bawah itu DAN biayanya berjam-jam,')
print('STOP -- Task 5 memang prioritas terendah (C). Head grid sudah lebih murah & menjanjikan.')

## Catatan

- Hasil di sini **belum** otomatis diserahkan ke Track C. Kalau suatu varian benar-benar
  mengalahkan baseline di CV 5-fold penuh, barulah lewati jalur handoff yang sama
  dengan notebook 07 (`handoff_v3.handoff_winner`, guard anti-overwrite).
- Aturan seleksi B9 tetap berlaku: `mean` naik tapi `min` antar-fold turun → **TOLAK**.
- Skor test 0.985 tetap **bukan** kriteria seleksi.